In [ ]:
import pointblank as pb
import polars as pl
import re

from pointblank import Validate, load_dataset
from pointblank.segments import Segment

### `seg_range()`

pb.seg_range() should work like `range()`

Examples:
`pb.seg_range(4) => Segment(segments=[[0, 1, 2, 3]])`

In [2]:
df = pb.load_dataset()

In [3]:
df

date_time,date,a,b,c,d,e,f
datetime[μs],date,i64,str,i64,f64,bool,str
2016-01-04 11:00:00,2016-01-04,2,"""1-bcd-345""",3,3423.29,true,"""high"""
2016-01-04 00:32:00,2016-01-04,3,"""5-egh-163""",8,9999.99,true,"""low"""
2016-01-05 13:32:00,2016-01-05,6,"""8-kdg-938""",3,2343.23,true,"""high"""
2016-01-06 17:23:00,2016-01-06,2,"""5-jdo-903""",null,3892.4,false,"""mid"""
2016-01-09 12:36:00,2016-01-09,8,"""3-ldm-038""",7,283.94,true,"""low"""
…,…,…,…,…,…,…,…
2016-01-20 04:30:00,2016-01-20,3,"""5-bce-642""",9,837.93,false,"""high"""
2016-01-20 04:30:00,2016-01-20,3,"""5-bce-642""",9,837.93,false,"""high"""
2016-01-26 20:07:00,2016-01-26,4,"""2-dmx-010""",7,833.98,true,"""low"""


In [4]:
def seg_range(start: int, stop: int | None = None, step: int = 1):
    if step == 0:
        raise ValueError("Step cannot be 0")
    if stop is None:
        start, stop = 0, start
    return Segment([list(range(start, stop, step))])

In [5]:
validation = (
    pb.Validate(df)
    .col_vals_gt(
        columns="d",
        value=800,
        segments=("a", seg_range(7))
    )
    .col_vals_expr(
        expr=pl.col("e").eq(True),
        segments=("c", seg_range(3, 9))
    )
    .col_vals_lt(
        columns="d",
        value=2000,
        segments=("a", seg_range(2, 10, 2))
    )
)

validation.interrogate()

Validate(data=shape: (13, 8)
┌─────────────────────┬────────────┬─────┬───────────┬──────┬─────────┬───────┬──────┐
│ date_time           ┆ date       ┆ a   ┆ b         ┆ c    ┆ d       ┆ e     ┆ f    │
│ ---                 ┆ ---        ┆ --- ┆ ---       ┆ ---  ┆ ---     ┆ ---   ┆ ---  │
│ datetime[μs]        ┆ date       ┆ i64 ┆ str       ┆ i64  ┆ f64     ┆ bool  ┆ str  │
╞═════════════════════╪════════════╪═════╪═══════════╪══════╪═════════╪═══════╪══════╡
│ 2016-01-04 11:00:00 ┆ 2016-01-04 ┆ 2   ┆ 1-bcd-345 ┆ 3    ┆ 3423.29 ┆ true  ┆ high │
│ 2016-01-04 00:32:00 ┆ 2016-01-04 ┆ 3   ┆ 5-egh-163 ┆ 8    ┆ 9999.99 ┆ true  ┆ low  │
│ 2016-01-05 13:32:00 ┆ 2016-01-05 ┆ 6   ┆ 8-kdg-938 ┆ 3    ┆ 2343.23 ┆ true  ┆ high │
│ 2016-01-06 17:23:00 ┆ 2016-01-06 ┆ 2   ┆ 5-jdo-903 ┆ null ┆ 3892.4  ┆ false ┆ mid  │
│ 2016-01-09 12:36:00 ┆ 2016-01-09 ┆ 8   ┆ 3-ldm-038 ┆ 7    ┆ 283.94  ┆ true  ┆ low  │
│ …                   ┆ …          ┆ …   ┆ …         ┆ …    ┆ …       ┆ …     ┆ …    │
│ 2016-01-20 04:30:00 ┆ 2016-01-20 ┆ 3   ┆ 5-bce-642 ┆ 9    ┆ 837.93  ┆ false ┆ high │
│ 2016-01-20 04:30:00 ┆ 2016-01-20 ┆ 3   ┆ 5-bce-642 ┆ 9    ┆ 837.93  ┆ false ┆ high │
│ 2016-01-26 20:07:00 ┆ 2016-01-26 ┆ 4   ┆ 2-dmx-010 ┆ 7    ┆ 833.98  ┆ true  ┆ low  │
│ 2016-01-28 02:51:00 ┆ 2016-01-28 ┆ 2   ┆ 7-dmx-010 ┆ 8    ┆ 108.34  ┆ false ┆ low  │
│ 2016-01-30 11:23:00 ┆ 2016-01-30 ┆ 1   ┆ 3-dka-303 ┆ null ┆ 2230.09 ┆ true  ┆ high │
└─────────────────────┴────────────┴─────┴───────────┴──────┴─────────┴───────┴──────┘, tbl_name=None, label=None, thresholds=Thresholds(warning=None, error=None, critical=None), actions=None, final_actions=None, brief=None, lang='en', locale='en')

In [6]:
tbl = pl.DataFrame(
    {
        "Name": ["Jeremy Cameron", "Jack Gunston", "Ben King", "Jamie Elliott", "Aaron Naughton", "Riley Thilthorpe", "Mitch Georgiades", "Logan Morris", "Sam Darcy", "Jack Higgins", "Jesse Hogan", "Ben Long", "Aaron Cadman", "Shannon Neale", "Josh Treacy",],
        "Position": ["Key Forward", "Key Forward", "Key Forward", "General Forward", "Key Forward", "Key Forward", "Key Forward", "Key Forward", "Key Forward", "General Forward", "Key Forward", "General Forward", "Key Forward", "Key Forward", "Key Forward"],
        "Goals": [88, 73, 71, 60, 60, 60, 58, 53, 48, 46, 46, 45, 44, 44, 44,],
        "Points": [43, 37, 23, 30, 27, 29, 42, 20, 21, 17, 14, 26, 23, 18, 18,],
        "Goal Assists": [15, 20, 8, 14, 19, 21, 7, 16, 10, 14, 6, 16, 12, 12, 12,],
        "Accuracy": [57.9, 56.6, 65.7, 56.6, 60, 63.8, 51.8, 60.2, 63.2, 63, 63.9, 53.6, 57.9, 64.7, 66.7,]
    }
)

In [7]:
tbl = pl.DataFrame(
    {
        "Name": ["Jeremy Cameron", "Jack Gunston", "Ben King", "Jamie Elliott", "Aaron Naughton", "Riley Thilthorpe", "Mitch Georgiades", "Logan Morris", "Sam Darcy", "Jack Higgins"],
        "Goals": [88, 73, 71, 60, 60, 60, 58, 53, 48, 46],
        "Accuracy": [57.9, 56.6, 65.7, 56.6, 60, 63.8, 51.8, 60.2, 63.2, 63]
    }
)

In [8]:
validation = (
    pb.Validate(tbl)
    .col_vals_gt("Accuracy", 55, segments=("Goals", seg_range(50, 61)))
    .interrogate()
)

validation

Validate(data=shape: (10, 3)
┌──────────────────┬───────┬──────────┐
│ Name             ┆ Goals ┆ Accuracy │
│ ---              ┆ ---   ┆ ---      │
│ str              ┆ i64   ┆ f64      │
╞══════════════════╪═══════╪══════════╡
│ Jeremy Cameron   ┆ 88    ┆ 57.9     │
│ Jack Gunston     ┆ 73    ┆ 56.6     │
│ Ben King         ┆ 71    ┆ 65.7     │
│ Jamie Elliott    ┆ 60    ┆ 56.6     │
│ Aaron Naughton   ┆ 60    ┆ 60.0     │
│ Riley Thilthorpe ┆ 60    ┆ 63.8     │
│ Mitch Georgiades ┆ 58    ┆ 51.8     │
│ Logan Morris     ┆ 53    ┆ 60.2     │
│ Sam Darcy        ┆ 48    ┆ 63.2     │
│ Jack Higgins     ┆ 46    ┆ 63.0     │
└──────────────────┴───────┴──────────┘, tbl_name=None, label=None, thresholds=Thresholds(warning=None, error=None, critical=None), actions=None, final_actions=None, brief=None, lang='en', locale='en')

In [15]:
from pointblank import Validate, load_dataset

In [20]:
load_dataset(dataset="small_table", tbl_type="polars")

date_time,date,a,b,c,d,e,f
datetime[μs],date,i64,str,i64,f64,bool,str
2016-01-04 11:00:00,2016-01-04,2,"""1-bcd-345""",3,3423.29,true,"""high"""
2016-01-04 00:32:00,2016-01-04,3,"""5-egh-163""",8,9999.99,true,"""low"""
2016-01-05 13:32:00,2016-01-05,6,"""8-kdg-938""",3,2343.23,true,"""high"""
2016-01-06 17:23:00,2016-01-06,2,"""5-jdo-903""",null,3892.4,false,"""mid"""
2016-01-09 12:36:00,2016-01-09,8,"""3-ldm-038""",7,283.94,true,"""low"""
…,…,…,…,…,…,…,…
2016-01-20 04:30:00,2016-01-20,3,"""5-bce-642""",9,837.93,false,"""high"""
2016-01-20 04:30:00,2016-01-20,3,"""5-bce-642""",9,837.93,false,"""high"""
2016-01-26 20:07:00,2016-01-26,4,"""2-dmx-010""",7,833.98,true,"""low"""


In [21]:
validation = (
    Validate(data=load_dataset(dataset="small_table", tbl_type="polars"))
    .col_vals_gt(
        columns="d",
        value=1000,
        segments=("a", seg_range(0, 10, 2)),
    )
    .interrogate()
)

validation

Validate(data=shape: (13, 8)
┌─────────────────────┬────────────┬─────┬───────────┬──────┬─────────┬───────┬──────┐
│ date_time           ┆ date       ┆ a   ┆ b         ┆ c    ┆ d       ┆ e     ┆ f    │
│ ---                 ┆ ---        ┆ --- ┆ ---       ┆ ---  ┆ ---     ┆ ---   ┆ ---  │
│ datetime[μs]        ┆ date       ┆ i64 ┆ str       ┆ i64  ┆ f64     ┆ bool  ┆ str  │
╞═════════════════════╪════════════╪═════╪═══════════╪══════╪═════════╪═══════╪══════╡
│ 2016-01-04 11:00:00 ┆ 2016-01-04 ┆ 2   ┆ 1-bcd-345 ┆ 3    ┆ 3423.29 ┆ true  ┆ high │
│ 2016-01-04 00:32:00 ┆ 2016-01-04 ┆ 3   ┆ 5-egh-163 ┆ 8    ┆ 9999.99 ┆ true  ┆ low  │
│ 2016-01-05 13:32:00 ┆ 2016-01-05 ┆ 6   ┆ 8-kdg-938 ┆ 3    ┆ 2343.23 ┆ true  ┆ high │
│ 2016-01-06 17:23:00 ┆ 2016-01-06 ┆ 2   ┆ 5-jdo-903 ┆ null ┆ 3892.4  ┆ false ┆ mid  │
│ 2016-01-09 12:36:00 ┆ 2016-01-09 ┆ 8   ┆ 3-ldm-038 ┆ 7    ┆ 283.94  ┆ true  ┆ low  │
│ …                   ┆ …          ┆ …   ┆ …         ┆ …    ┆ …       ┆ …     ┆ …    │
│ 2016-01-20 04:30:00 ┆ 2016-01-20 ┆ 3   ┆ 5-bce-642 ┆ 9    ┆ 837.93  ┆ false ┆ high │
│ 2016-01-20 04:30:00 ┆ 2016-01-20 ┆ 3   ┆ 5-bce-642 ┆ 9    ┆ 837.93  ┆ false ┆ high │
│ 2016-01-26 20:07:00 ┆ 2016-01-26 ┆ 4   ┆ 2-dmx-010 ┆ 7    ┆ 833.98  ┆ true  ┆ low  │
│ 2016-01-28 02:51:00 ┆ 2016-01-28 ┆ 2   ┆ 7-dmx-010 ┆ 8    ┆ 108.34  ┆ false ┆ low  │
│ 2016-01-30 11:23:00 ┆ 2016-01-30 ┆ 1   ┆ 3-dka-303 ┆ null ┆ 2230.09 ┆ true  ┆ high │
└─────────────────────┴────────────┴─────┴───────────┴──────┴─────────┴───────┴──────┘, tbl_name=None, label=None, thresholds=Thresholds(warning=None, error=None, critical=None), actions=None, final_actions=None, brief=None, lang='en', locale='en')